# Homework 1 - Information Retrieval.


Jeroen Baars - 10686320

Bob van den Hoogen - 10420169

Lea .

## Module 1: Evaluation.

Deadline: Wednesday, January 17th, midnight (23:59).

Collaboration: This is a team-based assignment. Form teams of 3 people.

Submit: An IPython Notebook for both the theoretical and the experimental parts with the necessary (a) answers, (b) implementation, (c) explanations,  (d) comments, and (e) analysis. Code quality, informative comments, detailed explanations of what each block in the notebook does and more important convincing analysis of the results will be considered when grading. 

Submit your work through Blackboard. All students in a team need to submit their work.

Filename: < student 1 id >–< student 2 id >–< student 3 id >–hw1.ipynb.

The homework will cover the following three topics covered in Lecture 1, 2, and 3:
- Evaluation measures;
- Interleaving;
- Click models.

The homework has a small theoretical part and a large experimental part.


## Theoretical Part [15 pts]

__Question 1.Hypothesis Testing – The problem of multiple comparisons [5 points]
Experimentation in AI often happens like this:__
- Modify/Build an algorithm
- Compare the algorithm to a baseline by running a hypothesis test.
- If not significant, go back to step A
- If significant, start writing a paper. 

Compute the probabilities below ~~How many hypothesis tests, m, does it take to get to~~ (with Type I error for each test = α):

__A. P(mth experiment gives significant result | m experiments lacking power to reject H0)?__

Answer:

If multiple tests are performed, after each other, the possibility of an event being true increases, caused by the chance of the event happening.  

P(at least one significant result) = $1 - (1 - x)^m$

Where m is the number of experiments performed. x is the significance level. For a one-tailed test this is $\alpha$. For a two-tailed test this is $(\alpha /2)$.

mogelijk iets:
https://www.stat.berkeley.edu/~mgoldman/Section0402.pdf

piazza staat de vraag anders geformuleerd en uitgelegd:

The chance of getting a Type I error is present in all experiments and independently it is $\alpha$ We care about the m-th experiment making an actual mistake. Here is a reformulation of the question:

 

Suppose hypothetically that the null hypothesis is actually true.

 

The probability of concluding it is false after one test is α. If we do not find it false, we run a second experiment.

What is the probability of concluding that it is false in this second experiment? If we do not find it false, we run a third experiment. And so on and so forth. If we do not find it wrong after m-1 experiment, we run an m-th one. What is the probability of concluding that it is false in this m-th experiment?



__B. P(at least one significant result | m experiments lacking power to reject H0)?__

Answer:


__Question 2. Bias and unfairness in Interleaving experiments [10 points].__

Balance interleaving has been shown to be biased in a number of corner cases. An example was given during the lecture with two ranked lists of length 3 being interleaved, and a randomly clicking population of users that resulted in algorithm A winning ⅔ of the time, even though in theory the percentage of wins should be 50% for both algorithms. Can you come up with a situation of two ranked lists of length 3 and a distribution of clicks over them for which Team-draft interleaving is unfair to the better algorithm?

Answer:




## Experimental Part [85 pts]

In [1]:
import itertools
import numpy as np
import random


Commercial search engines use both offline and online approach in evaluating a new search algorithm: they first use an offline test collection to compare the production algorithm (P) with the new experimental algorithm (E); if E statistically significantly outperforms P with respect to the evaluation measure of their interest, the two algorithms are then compared online through an interleaving experiment.

For the purpose of this homework we will assume that the evaluation measures of interest are:
- Binary evaluation measures
    - Precision at rank k,
    - Recall at rank k,
    - Average Precision,
- Multi-graded evaluation measures
    - Normalized Discounted Cumulative Gain at rank k (nDCG@k),
    - Expected Reciprocal Rank (ERR).
    
Further, for the purpose of this homework we will assume that the interleaving algorithms of interest are:
- Team-Draft Interleaving (Joachims. "Evaluating retrieval performance using clickthrough data". Text Mining 2003.),
- ~~Probabilistic Interleaving (Hofmann, Whiteson, and de Rijke. "A probabilistic method for inferring preferences from clicks." CIKM 2011.).~~
 
In an interleaving experiment the ranked results of P and E (against a user query) are interleaved in a single ranked list which is presented to a user. The user then clicks on the results and the algorithm that receives most of the clicks wins the comparison. The experiment is repeated for a number of times (impressions) and the total wins for P and E are computed. 

A Sign/Binomial Test is then run to examine whether the difference in wins between the two algorithms is statistically significant (or due to chance). Alternatively one can calculate the proportion of times the E wins and test whether this proportion, p, is greater than p0=0.5. This is called an 1-sample 1-sided proportion test.

One of the key questions however is __whether offline evaluation and online evaluation outcomes agree with each other__. In this homework you will determine the degree of agreement between offline evaluation measures and interleaving outcomes, by the means of simulations. A similar analysis using actual online data can be found at Chapelle et al. “Large-Scale Validation and Analysis of Interleaved Search Evaluation”.


### [Based on Lecture 1]

__Step 1: Simulate Rankings of Relevance for E and P (5 points).__

In the first step you will generate pairs of rankings of relevance, for the production P and experimental E, respectively, for a hypothetical query q. Assume a 3-graded relevance, i.e. {N, R, HR}. Construct all possible P and E ranking pairs of length 5.<br>

This step should give you about.<br>
Example:<br>
P: {N N N N N}<br>
E: {N N N N R}<br>
…<br>
P: {HR HR HR HR R}<br>
E: {HR HR HR HR HR}<br>

(Note 1: If you do not have enough computational power, sample 5000 pair uniformly at random to show your work.)


In [2]:
#with the itertools product function create all possibile combinations with length 10 and N R HR as possible input
combis = list(itertools.product(['N','R','HR'], repeat=10))
#Create a list where the list of length 10 is split into 2 seperate lists of length 5
#Also don't include the lists where all both lists are equal
combinations = [[comb[:5],comb[5:]] for comb in combis if comb[:5] != comb[5:]]

In [3]:
#Print some combinations as an example
for i in combinations[0:3]:
    print(i)
for j in combinations[len(combinations)-3:]:
    print(j)

[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'N', 'R')]
[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'N', 'HR')]
[('N', 'N', 'N', 'N', 'N'), ('N', 'N', 'N', 'R', 'N')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'R', 'HR')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'HR', 'N')]
[('HR', 'HR', 'HR', 'HR', 'HR'), ('HR', 'HR', 'HR', 'HR', 'R')]


__Step 2: Implement Evaluation Measures (10 points).__

Implement 1 binary and 2 multi-graded evaluation measures out of the 7 measures mentioned above. 
(Note 2: Some of the aforementioned measures require the total number of relevant and highly relevant documents in the entire collection – pay extra attention on how to find this)


In [4]:
#Precision at rank 3
#We researched precision at rank 3 and looped through a subset of 20.000
#We took the subset of 20.000 to create a difference between algorithm P and E in precision

#parameters to save amount of HR or R in the query
relevantP = 0
relevantE = 0

#Loop through al combinations and add up amount of HR and R
for pair in combinations[0:20000]:
    for doc in pair[0][0:3]:
        if doc == 'R' or doc == 'HR':
            relevantP +=1
    for doc in pair[1][0:3]:
        if doc == 'R' or doc == 'HR':
            relevantE +=1

#Calculate the precision for both algorithms
precisionP = relevantP/(len(combinations)*3)
precisionE = relevantE/(len(combinations)*3)

#Print the precision for both algorithms
print("precision of p is : ", precisionP)
print("precision of e is : ", precisionE)

precision of p is :  0.15040415377115715
precision of e is :  0.22687027400832116


In [5]:
# Normalized Discounted Cumulative Gain at rank 3
# For comparsion we took the same subset of 20.000
ranks = {'N': 0, 'R': 1, 'HR': 2}

NDCG_list_p = []
NDCG_list_e = []

for pair in combinations[0:20000]:
    DCG_p = 0
    IDCG_p = 0
    DCG_e = 0
    IDCG_e = 0
    
    ideal_p = []
    for i, doc in enumerate(pair[0][0:3]):      
        CG_p = ranks[doc]
        ideal_p.append(CG_p)
        DCG_p += (2 ** CG_p - 1) / (np.log(i + 2.0)/ np.log(2.0))
    ideal_p.sort(reverse=True)
    for i, CG in enumerate(ideal_p):
        IDCG_p += (2 ** CG - 1) / (np.log(i + 2.0)/ np.log(2.0))
        NDCG_p = 0 if IDCG_p == 0 else DCG_p / IDCG_p
    NDCG_list_p.append(NDCG_p)
        
    ideal_e = []
    for i, doc in enumerate(pair[1][0:3]):      
        CG_e = ranks[doc]
        ideal_e.append(CG_e)
        DCG_e += (2 ** CG_e - 1) / (np.log(i + 2.0)/ np.log(2.0))
    ideal_e.sort(reverse=True)
    for i, CG in enumerate(ideal_e):
        IDCG_e += (2 ** CG - 1) / (np.log(i + 2.0)/ np.log(2.0))
        NDCG_e = 0 if IDCG_e == 0 else DCG_e / IDCG_e
    NDCG_list_e.append(NDCG_e)

print('average NDCG of p is: ', np.average(NDCG_list_p))
print('average NDCG of e is: ', np.average(NDCG_list_e))

average NDCG of p is:  0.552921590498
average NDCG of e is:  0.797606070489


In [6]:
ERRe = 0
ERRp = 0

PN = 0
PR = 1/4
PHR = 3/4

Pstop = [0,0,0,0,0]
ERR = [0,0,0,0,0]

def check_p(doc):
    if doc == 'N':
        return PN
    elif doc == 'R':
        return PR
    elif doc == 'HR':
        return PHR

ERR = []

for pair in combinations[0:20000]:
    
    ERRsum = [0,0]
    ERRpartial = [0,0,0,0,0]
    
    for index, query in enumerate(pair):
        for i, doc in enumerate(query):
            Pstop[i] = check_p(doc)
        for j, p in enumerate(Pstop):
            for k in range(j):
                if k == 0:
                    sumup = 1-Pstop[k]
                else:
                    sumup *= (1/k)*(1-Pstop[k])
            if j == 0:
                total = Pstop[j]
            else:
                total = (1/(j+1)) * Pstop[j] * sumup
            ERRpartial[j] = total
        ERRsum[index] = sum(ERRpartial)
    ERR.append(ERRsum)

deltaERR = []

ERRp = 0
ERRe = 0
for pair in ERR:
    ERRp += pair[0]
    ERRe += pair[1]
    
print('average ERR of p is: ', ERRp/20000)
print('average ERR of e is: ', ERRe/20000)

average ERR of p is:  0.2623504166666839
average ERR of e is:  0.5081259391276026


__Step 3: Calculate the 𝛥measure (0 points).__

For the three measures and all P and E ranking pairs constructed above calculate the difference: 𝛥measure = measureE-measureP. Consider only those pairs for which E outperforms P.


In [7]:
#Precision

winsE = 0

for pair in combinations[0:20000]:
    relevantP = 0
    relevantE = 0
    for doc in pair[0][0:3]:
        if doc == 'R' or doc == 'HR':
            relevantP +=1
    for doc in pair[1][0:3]:
        if doc == 'R' or doc == 'HR':
            relevantE +=1
    if relevantE > relevantP:
        winsE += 1

print("In precision, of 20.000, e wins:", winsE)
            

# NCDG
d_NDCG = []
d_NDCG_ix = []
for i in range(len(NDCG_list_e)):
    if NDCG_list_e[i] > NDCG_list_p[i]:
        d_NDCG.append(NDCG_list_e[i] - NDCG_list_p[i])
        d_NDCG_ix.append(i)
        
print("In NDCG, of 20.000, e wins:", len(d_NDCG))

#ERR
deltaERR = []
for ind, comb in enumerate(ERR):
    if comb[1] > comb[0]:
        deltaERR.append([ind, comb[1]-comb[0]])

print("In ERR, of 20.000, e wins:", len(deltaERR))



In precision, of 20.000, e wins: 11461
In NDCG, of 20.000, e wins: 15633
In ERR, of 20.000, e wins: 15727


### [Based on Lecture 2]
__Step 4: Implement Interleaving (15 points).__

Implement ~~2~~ 1 interleaving algorithms: (1) Team-Draft Interleaving OR Balanced Interleaving, ~~AND (2), Probabilistic Interleaving.~~ The interleaving algorithms should (a) given two rankings of relevance interleave them into a single ranking, and (b) given the users clicks on the interleaved ranking assign credit to the algorithms that produced the rankings.

(Note 4: Note here that as opposed to a normal interleaving experiment where rankings consists of urls or docids, in our case the rankings consist of relevance labels. Hence in this case (a) you will assume that E and P return different documents, (b) the interleaved ranking will also be a ranking of labels.)


In [8]:
def Interleaving(rank_p, rank_e):
    toincoss = np.round(np.random.rand())
    interleaved_rank = []
    interleaved_docs = []
    for i in range(len(rank_p)):
        if i % 2 == toincoss:
            interleaved_rank.append(rank_p[i])
            interleaved_docs.append('prod')
        else:
            interleaved_rank.append(rank_e[i])
            interleaved_docs.append('exp')
    return interleaved_rank, interleaved_docs

def Interleaving_evaluation(clicks, interleaved_docs):
    score_p = 0
    score_e = 0
    for click in clicks:
        if interleaved_docs[click] == 'prod':
            score_p += 1
        else:
            score_e += 1
    return score_p, score_e
            

Assuming that the clicks will be given in as list with indices of the clicked articles, calling the Interleaving_evaluation function would return a score for both production and experimental.

#### [Based on Lecture 3]
### Step 5: Implement User Clicks Simulation (15 points).

Having interleaved all the ranking pairs an online experiment could be ran. However, given that we do not have any users (and the entire homework is a big simulation) we will simulate user clicks.
We have considered a number of click models including:

- Random Click Model (RCM)
- Position-Based Model (PBM)
- Simple Dependent Click Model (SDCM)
- Simple Dynamic Bayesian Network (SDBN) <br>

Consider two different click models, (a) the Random Click Model (RCM), and (b) one out of the remaining 3 aforementioned models. The parameters of some of these models can be estimated using the Maximum Likelihood Estimation (MLE) method, while others require using the Expectation-Maximization (EM) method. 

Implement the two models so that 
- (a) there is a method that learns the parameters of the model given a set of training data, 
- (b) there is a method that predicts the click probability given a ranked list of relevance labels, 
- (c) there is a method that decides - stochastically - whether a document is clicked based on these probabilities.
<br>

Having implemented the two click models, estimate the model parameters using the Yandex Click Log [file].
(Note 6: Do not learn the attractiveness parameter $\alpha_{uq}$)

https://drive.google.com/file/d/1tqMptjHvAisN1CJ35oCEZ9_lb0cEJwV0/view


In [9]:
file = open("YandexRelPredChallenge.txt", "r")

# Random Click Model

query = 0
click = 0

for line in file: 
    line = str(line)
    line = line.split("\t")
    if line[2] == 'Q':
        query += 1 
    elif line[2] == 'C':
        click += 1

alpha = click / (query * 10)

def flip(alpha):
    """
    Stochastic clicking model
    """
    return 'C' if random.random() < alpha else 'NC'

N = 100000
flips = [flip(alpha) for i in range(N)]

print("amount of clicked in 100000 documents:", float(flips.count('C'))/N)


amount of clicked in 100000 documents: 0.13654


In [42]:
# SDCM

file = open("YandexRelPredChallenge.txt", "r")

def flip2(alpha):
    """
    Stochastic clicking model
    """
    return random.random() < alpha

clicked_r = np.zeros(10)
last_clicked = np.zeros(10)
cur_query = []
last_click = ''
click = False

for i, line in enumerate(file):
    line = str(line)
    line = line.split("\n")[0]
    line = line.split("\t")
    
    # when query follows query
    if line[2] == 'Q' and click == False:
        cur_query = line[5:15]
    
    # when query follows click
    if line[2] == 'Q' and click == True:
        try:
            last_clicked[cur_query.index(str(last_click))] += 1
            click = False
            cur_query = line[5:15]
        except ValueError:
            click = False
            cur_query = line[5:15]
    
    # we get a click
    if line[2] == 'C':
        click = True
        last_click = line[3]
        try:
            clicked_r[cur_query.index(str(line[3]))] += 1
        except ValueError:
            None


satisfaction_r = last_clicked / clicked_r
continuation_r = 1 - satisfaction_r

interleavedCombinationsSDCM = []

for [pair1, pair2] in combinations[:20000]:
    interleavedCombinationsSDCM.append(Interleaving(pair1, pair2))

clickCombinationsSDCM = []
for comb in interleavedCombinationsSDCM:
    clicked = ['NC' for i in range(5)]
    for i, doc in enumerate(comb[1]):
        clicked[i] = flip(alpha)
        if clicked[i] == 'C' and flip2(satisfaction_r[i]):
            break
    clickCombinationsSDCM.append([comb[0],comb[1],clicked])

In [14]:
# random click model

interleavedCombinations = []

for [pair1, pair2] in combinations[:20000]:
    interleavedCombinations.append(Interleaving(pair1, pair2))
    
clickCombinations = []

for comb in interleavedCombinations:
    clicked = []
    for doc in comb[1]:
        clicked.append(flip(alpha))
        clickCombinations.append([comb[0],comb[1],clicked])



In [44]:
print(interleavedCombinationsSDCM[0:10], '\n')
print(clickCombinationsSDCM[0:10])

[(['N', 'N', 'N', 'N', 'R'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'N', 'N'], ['prod', 'exp', 'prod', 'exp', 'prod']), (['N', 'N', 'N', 'N', 'N'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'N', 'R'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'N', 'HR'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'N', 'N'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'HR', 'N'], ['prod', 'exp', 'prod', 'exp', 'prod']), (['N', 'N', 'N', 'N', 'HR'], ['exp', 'prod', 'exp', 'prod', 'exp']), (['N', 'N', 'N', 'N', 'N'], ['prod', 'exp', 'prod', 'exp', 'prod']), (['N', 'N', 'N', 'N', 'N'], ['prod', 'exp', 'prod', 'exp', 'prod'])] 

[[['N', 'N', 'N', 'N', 'R'], ['exp', 'prod', 'exp', 'prod', 'exp'], ['NC', 'NC', 'NC', 'NC', 'NC']], [['N', 'N', 'N', 'N', 'N'], ['prod', 'exp', 'prod', 'exp', 'prod'], ['NC', 'NC', 'NC', 'NC', 'NC']], [['N', 'N', 'N', 'N', 'N'], ['exp', 'prod', 'exp', 'prod', 'exp'], ['NC', 'C', 'NC', 'NC', 'NC']], [['N', 'N